# Assignment 2b: Edge Detection, Noise, and Denoising

Run from the assignment folder. This notebook reads `assignment-1-data/blox.jpg` and saves edge, noise, and denoising results under `output/part2b_edges/`.


In [ ]:
%matplotlib inline

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


## Part 2b: Edge Detection, Noise, and Denoising

Load blox.jpg as grayscale. Compare Sobel gradient magnitude, Canny edges, inverse intensity thresholding, and a gray-level range mask. Then add zero-mean Gaussian noise at 20 dB and 10 dB using the assignment PSNR equation. Noise arrays remain floating point and unclipped so their requested standard deviations are preserved.

In [ ]:
edge_input_path = Path('assignment-1-data/blox.jpg')
edge_output_dir = Path('output/part2b_edges')
edge_output_dir.mkdir(parents=True, exist_ok=True)

blox_bgr = cv2.imread(str(edge_input_path), cv2.IMREAD_COLOR)
if blox_bgr is None:
    raise FileNotFoundError(f'Could not read {edge_input_path}')
blox_gray = cv2.cvtColor(blox_bgr, cv2.COLOR_BGR2GRAY)
print(f'BloX grayscale range: {blox_gray.min()} to {blox_gray.max()}')


In [ ]:
def prepare_canny_input(image, low_threshold=50, high_threshold=150):
    """Convert an image to uint8; scale Canny thresholds with any affine rescale."""
    array = np.asarray(image)
    if array.dtype == np.uint8:
        return array, int(low_threshold), int(high_threshold), 1.0
    array = array.astype(np.float32)
    image_min = float(array.min())
    image_max = float(array.max())
    scale = 255.0 / (image_max - image_min) if image_max > image_min else 1.0
    canny_image = np.rint((array - image_min) * scale).clip(0, 255).astype(np.uint8)
    return (
        canny_image,
        max(1, int(round(low_threshold * scale))),
        max(1, int(round(high_threshold * scale))),
        scale,
    )


def analyze_edges(image):
    """Run Sobel, Canny, inverse threshold, and gray-band extraction."""
    gray_float = np.asarray(image, dtype=np.float32)
    sobel_x = cv2.Sobel(gray_float, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray_float, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(sobel_x, sobel_y)
    _, sobel_binary = cv2.threshold(magnitude, 110, 255, cv2.THRESH_BINARY)
    canny_input, low, high, scale = prepare_canny_input(image)
    canny_edges = cv2.Canny(canny_input, low, high)
    _, dark_regions = cv2.threshold(gray_float, 165, 255, cv2.THRESH_BINARY_INV)
    gray_band = cv2.inRange(gray_float, 60, 160)
    return {
        'sobel_x': sobel_x,
        'sobel_y': sobel_y,
        'magnitude': magnitude,
        'sobel_binary': sobel_binary.astype(np.uint8),
        'canny': canny_edges,
        'canny_input': canny_input,
        'dark_regions': dark_regions.astype(np.uint8),
        'gray_band': gray_band,
        'canny_thresholds': (low, high),
        'canny_scale': scale,
    }


def magnitude_preview(magnitude):
    peak = float(np.max(magnitude))
    if peak <= 0:
        return np.zeros(magnitude.shape, dtype=np.uint8)
    return np.rint(magnitude * (255.0 / peak)).clip(0, 255).astype(np.uint8)


def save_edge_result(name, result):
    cv2.imwrite(str(edge_output_dir / f'{name}_sobel_magnitude.png'), magnitude_preview(result['magnitude']))
    cv2.imwrite(str(edge_output_dir / f'{name}_sobel_binary.png'), result['sobel_binary'])
    cv2.imwrite(str(edge_output_dir / f'{name}_canny.png'), result['canny'])
    cv2.imwrite(str(edge_output_dir / f'{name}_dark_regions.png'), result['dark_regions'])
    cv2.imwrite(str(edge_output_dir / f'{name}_gray_band.png'), result['gray_band'])
    cv2.imwrite(str(edge_output_dir / f'{name}_canny_input.png'), result['canny_input'])


clean_edges = analyze_edges(blox_gray)
save_edge_result('blox', clean_edges)
cv2.imwrite(str(edge_output_dir / 'blox_grayscale.png'), blox_gray)
print(f'Clean Canny edge coverage: {100*np.count_nonzero(clean_edges["canny"])/clean_edges["canny"].size:.1f}%')
print(f'Clean Sobel threshold coverage: {100*np.count_nonzero(clean_edges["sobel_binary"])/clean_edges["sobel_binary"].size:.1f}%')

fig, axes = plt.subplots(1, 6, figsize=(19, 4))
panels = [
    (blox_gray, 'Grayscale'),
    (magnitude_preview(clean_edges['magnitude']), 'Sobel magnitude'),
    (clean_edges['sobel_binary'], 'Sobel threshold'),
    (clean_edges['canny'], 'Canny'),
    (clean_edges['dark_regions'], 'Threshold < 165'),
    (clean_edges['gray_band'], 'inRange 60-160'),
]
for axis, (pixels, title) in zip(axes, panels):
    axis.imshow(pixels, cmap='gray', vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()


### Add white Gaussian noise at the requested PSNR levels

The exercise defines PSNR as 20 log10((Imax - Imin) / sigma_n), so sigma_n is the image range divided by 10^(PSNR/20). The following cell uses NumPy normal noise with mean zero and reports the realized sample standard deviation and corresponding PSNR. The unclipped float arrays are saved as NPY files because an 8-bit image cannot represent values outside 0-255 without clipping.


In [ ]:
image_range = float(blox_gray.max() - blox_gray.min())
np.random.seed(644)
noise_images = {}
noise_edge_results = {}

for target_psnr in (20, 10):
    sigma = image_range / (10 ** (target_psnr / 20.0))
    noise = np.random.normal(0.0, sigma, size=blox_gray.shape).astype(np.float32)
    noisy_float = blox_gray.astype(np.float32) + noise
    measured_sigma = float(noise.std())
    measured_psnr = 20 * np.log10(image_range / measured_sigma)
    label = f'{target_psnr}dB'
    noise_images[label] = noisy_float
    np.save(edge_output_dir / f'blox_noisy_{label}_float.npy', noisy_float)
    noise_edge_results[label] = analyze_edges(noisy_float)
    save_edge_result(f'blox_noisy_{label}', noise_edge_results[label])
    preview = np.clip(np.rint(noisy_float), 0, 255).astype(np.uint8)
    cv2.imwrite(str(edge_output_dir / f'blox_noisy_{label}.png'), preview)
    print(
        f'{label}: target sigma={sigma:.2f}, sampled sigma={measured_sigma:.2f}, '
        f'measured PSNR={measured_psnr:.2f} dB; Canny thresholds='
        f'{noise_edge_results[label]["canny_thresholds"]}'
    )

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, label in enumerate(('20dB', '10dB')):
    result = noise_edge_results[label]
    display_noise = np.clip(np.rint(noise_images[label]), 0, 255).astype(np.uint8)
    panels = [
        (display_noise, f'Noisy input {label}'),
        (magnitude_preview(result['magnitude']), 'Sobel magnitude'),
        (result['sobel_binary'], 'Sobel threshold'),
        (result['canny'], 'Canny'),
    ]
    for axis, (pixels, title) in zip(axes[row], panels):
        axis.imshow(pixels, cmap='gray', vmin=0, vmax=255)
        axis.set_title(title)
        axis.axis('off')
plt.tight_layout()
plt.show()


### Denoising comparisons

Compare Gaussian smoothing with three kernel sizes, median filtering with two sizes, and bilateral filtering. Gaussian and bilateral filtering run on the floating-point noisy image. The median function below uses sliding windows so it can retain floating-point values and avoid clipping the noise before filtering. Sobel/Canny and the two intensity masks are then rerun for every denoised result. Printed Canny coverage allows comparison with the clean and noisy cases.


In [ ]:
def median_filter_float(image, size):
    """Median filter a 2D float image with reflected boundaries."""
    radius = size // 2
    padded = np.pad(image, ((radius, radius), (radius, radius)), mode='reflect')
    windows = np.lib.stride_tricks.sliding_window_view(padded, (size, size))
    return np.median(windows, axis=(-2, -1)).astype(np.float32)


def denoise_float_image(noisy):
    """Return several denoised float images and the settings used."""
    denoised = {}
    for size in (3, 7, 11):
        denoised[f'Gaussian {size}x{size}'] = cv2.GaussianBlur(
            noisy.astype(np.float32), (size, size), 0
        )
    for size in (3, 7):
        denoised[f'Median {size}x{size}'] = median_filter_float(noisy, size)
    denoised['Bilateral d=11'] = cv2.bilateralFilter(
        noisy.astype(np.float32), d=11, sigmaColor=75, sigmaSpace=75
    )
    return denoised


denoised_results = {}
denoised_edge_results = {}
for label in ('20dB', '10dB'):
    denoised_results[label] = denoise_float_image(noise_images[label])
    denoised_edge_results[label] = {}
    for method, denoised in denoised_results[label].items():
        safe_method = (
            method.lower().replace(' ', '_').replace('x', 'x').replace('=', '')
        )
        stem = f'blox_{label}_{safe_method}'
        np.save(edge_output_dir / f'{stem}_float.npy', denoised)
        preview = np.clip(np.rint(denoised), 0, 255).astype(np.uint8)
        cv2.imwrite(str(edge_output_dir / f'{stem}.png'), preview)
        edge_result = analyze_edges(denoised)
        denoised_edge_results[label][method] = edge_result
        save_edge_result(stem, edge_result)
        coverage = 100 * np.count_nonzero(edge_result['canny']) / edge_result['canny'].size
        print(f'{label} {method}: Canny edge coverage {coverage:.1f}%')

for label in ('20dB', '10dB'):
    methods = list(denoised_results[label])
    fig, axes = plt.subplots(2, len(methods) + 1, figsize=(3.1 * (len(methods) + 1), 6))
    input_preview = np.clip(np.rint(noise_images[label]), 0, 255).astype(np.uint8)
    axes[0, 0].imshow(input_preview, cmap='gray', vmin=0, vmax=255)
    axes[0, 0].set_title(f'Noisy {label}')
    axes[1, 0].imshow(noise_edge_results[label]['canny'], cmap='gray', vmin=0, vmax=255)
    axes[1, 0].set_title('Noisy Canny')
    for column, method in enumerate(methods, start=1):
        denoised = denoised_results[label][method]
        edge_result = denoised_edge_results[label][method]
        axes[0, column].imshow(np.clip(np.rint(denoised), 0, 255), cmap='gray', vmin=0, vmax=255)
        axes[0, column].set_title(method)
        axes[1, column].imshow(edge_result['canny'], cmap='gray', vmin=0, vmax=255)
        axes[1, column].set_title('Canny')
    for axis in axes.flat:
        axis.axis('off')
    fig.suptitle(f'Denoising and Canny comparison at {label}')
    plt.tight_layout()
    plt.show()


### Part 2b observations

The clean BloX image produces coherent object boundaries. Sobel magnitude shows gradient strength, while thresholding the magnitude keeps stronger gradients. Canny produces a thinner edge map. The inverse threshold and inRange outputs select intensity regions rather than edges.

Noise creates additional gradients and false Canny edges; the 10 dB case is noisier than the 20 dB case. Gaussian and median filters reduce texture as their window sizes increase, but strong smoothing also removes fine boundaries. Bilateral filtering smooths while retaining some larger edges. The notebook prints the Canny edge coverage for clean, noisy, and denoised images so these changes can be compared with the displayed maps. Float noise and denoised arrays are also saved for inspection.
